# Model Comparison & Product Assessment

**FPL Points Prediction: XGBoost vs LLM Approaches**

---

## Executive Summary

**The question:** Can a fine-tuned LLM compete with XGBoost for predicting Fantasy Premier League points?

**The answer:** Not on accuracy -- but it adds value through ranking and explanation.

We evaluated five models on 243 test-set players from Gameweek 31+:

| Model | MAE | Key Insight |
|-------|-----|-------------|
| **XGBoost** | **2.11** | Best accuracy, <1s inference, production-ready |
| Few-Shot (3 examples) | 2.16 | 97% of XGBoost accuracy with zero training |
| Chain-of-Thought | 2.20 | Reasoning visible, but no accuracy gain |
| Fine-Tuned v3 | 3.35 | Wider prediction spread aids ranking decisions |
| Zero-Shot | 11.54 | Without examples, the LLM hallucinates point scales |

The few-shot LLM is the surprise story: three examples in a prompt get you 97% of the way to a trained gradient boosting model. The fine-tuned model struggled with mode collapse across three training iterations, but its wider output range (6 distinct values from 0 to 10) makes it arguably better for *ranking* players -- the actual FPL decision task.

**Recommendation:** A hybrid system -- XGBoost for point estimation, LLM for explanations and edge-case reasoning -- is the optimal product.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# --- Paths ---
ROOT = Path("__file__").resolve().parent.parent
# Fallback if running from notebooks/
if not (ROOT / "eval").exists():
    ROOT = Path(".").resolve().parent
if not (ROOT / "eval").exists():
    ROOT = Path(".").resolve()

EVAL = ROOT / "eval"
DATA = ROOT / "data" / "processed"

# --- Style ---
plt.rcParams.update({
    "figure.facecolor": "#fafafa",
    "axes.facecolor": "#fafafa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.color": "#999999",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "figure.dpi": 120,
})

# Professional color palette
COLORS = {
    "xgboost": "#2563eb",       # blue
    "few_shot": "#16a34a",      # green
    "chain_of_thought": "#d97706",  # amber
    "fine_tuned": "#9333ea",    # purple
    "zero_shot": "#dc2626",     # red
}

MODEL_LABELS = {
    "xgboost": "XGBoost",
    "few_shot": "Few-Shot",
    "chain_of_thought": "Chain-of-Thought",
    "fine_tuned": "Fine-Tuned v3",
    "zero_shot": "Zero-Shot",
}

print(f"Project root: {ROOT}")
print(f"Eval dir exists: {EVAL.exists()}")

In [ ]:
# --- Load all data ---

with open(EVAL / "xgboost_results.json") as f:
    xgb_results = json.load(f)

with open(EVAL / "llm_results_summary.json") as f:
    llm_results = json.load(f)

# Load per-prediction files for detailed analysis
predictions = {}
for strategy in ["few_shot", "chain_of_thought", "fine_tuned", "zero_shot"]:
    with open(EVAL / f"llm_{strategy}_predictions.json") as f:
        preds = json.load(f)
    predictions[strategy] = pd.DataFrame(preds)

# Load features for the test set (GW 31+)
features = pd.read_csv(DATA / "features.csv")
test_set = features[features["gameweek"] > 30].reset_index(drop=True)

print(f"Test set: {len(test_set)} players")
print(f"Positions: {test_set['position'].value_counts().to_dict()}")

# Generate XGBoost predictions by re-running the model
import xgboost as xgb

model = xgb.XGBRegressor()
model.load_model(str(ROOT / "models" / "xgboost_fpl.json"))

DROP_COLS = ["player_id", "player_name", "target_points", "gameweek"]
train_set = features[features["gameweek"] <= 30]
X_train_cols = pd.get_dummies(train_set.drop(columns=DROP_COLS), columns=["position"]).columns

X_test = pd.get_dummies(test_set.drop(columns=DROP_COLS), columns=["position"])
X_test = X_test.reindex(columns=X_train_cols, fill_value=0)

xgb_preds = model.predict(X_test)
xgb_actual = test_set["target_points"].values

predictions["xgboost"] = pd.DataFrame({
    "index": range(len(xgb_preds)),
    "actual": xgb_actual,
    "parsed_prediction": xgb_preds,
})

print(f"XGBoost predictions loaded: {len(xgb_preds)}")
print(f"XGBoost MAE check: {np.mean(np.abs(xgb_preds - xgb_actual)):.4f}")

---

## 1. Full Results Table

All five models evaluated on the same 243-player test set (Gameweek 31+).

In [ ]:
# Build the comparison table
rows = []

# XGBoost
rows.append({
    "Model": "XGBoost",
    "MAE": xgb_results["mae"],
    "RMSE": xgb_results["rmse"],
    "Within +/-1": f"{xgb_results['within_1']:.1%}",
    "Within +/-3": f"{xgb_results['within_3']:.1%}",
    "Inference Time": "<1s",
    "Training Required": "Yes (500 trees)",
})

# LLM strategies
strategy_meta = {
    "few_shot": ("Few-Shot", "No"),
    "chain_of_thought": ("Chain-of-Thought", "No"),
    "fine_tuned": ("Fine-Tuned v3", "Yes (LoRA x3)"),
    "zero_shot": ("Zero-Shot", "No"),
}

for key, (label, training) in strategy_meta.items():
    r = llm_results[key]
    rows.append({
        "Model": label,
        "MAE": r["mae"],
        "RMSE": r["rmse"],
        "Within +/-1": f"{r['within_1']:.1%}",
        "Within +/-3": f"{r['within_3']:.1%}",
        "Inference Time": f"{r['time_seconds']:.0f}s",
        "Training Required": training,
    })

results_df = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)

# Style the table
def highlight_best(s):
    if s.name in ["MAE", "RMSE"]:
        best = s.min()
        return ["font-weight: bold; color: #2563eb" if v == best else "" for v in s]
    return ["" for _ in s]

styled = (results_df.style
    .apply(highlight_best, axis=0)
    .set_properties(**{"text-align": "center"})
    .set_properties(subset=["Model"], **{"text-align": "left", "font-weight": "bold"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center"), ("background-color", "#f1f5f9"), ("font-weight", "bold")]},
        {"selector": "td", "props": [("padding", "8px 12px")]},
    ])
    .hide(axis="index")
)
styled

---

## 2. MAE Comparison Chart

Horizontal bar chart showing all five models ranked by Mean Absolute Error.

In [ ]:
# MAE comparison - horizontal bar chart
models_sorted = results_df.sort_values("MAE", ascending=True)
model_names = models_sorted["Model"].values
mae_values = models_sorted["MAE"].values

color_map = {
    "XGBoost": COLORS["xgboost"],
    "Few-Shot": COLORS["few_shot"],
    "Chain-of-Thought": COLORS["chain_of_thought"],
    "Fine-Tuned v3": COLORS["fine_tuned"],
    "Zero-Shot": COLORS["zero_shot"],
}
bar_colors = [color_map[m] for m in model_names]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(model_names)), mae_values, color=bar_colors, height=0.6, edgecolor="white", linewidth=1.5)
ax.set_yticks(range(len(model_names)))
ax.set_yticklabels(model_names, fontsize=12, fontweight="bold")
ax.invert_yaxis()
ax.set_xlabel("Mean Absolute Error (lower is better)", fontsize=12)
ax.set_title("Model Accuracy Comparison", fontsize=16, fontweight="bold", pad=15)

# Add value labels
for bar, val in zip(bars, mae_values):
    ax.text(val + 0.15, bar.get_y() + bar.get_height()/2, f"{val:.2f}",
            va="center", ha="left", fontsize=11, fontweight="bold", color="#333")

# Add a vertical reference line at XGBoost MAE
ax.axvline(mae_values[0], color=COLORS["xgboost"], linestyle="--", alpha=0.4, linewidth=1)

ax.set_xlim(0, max(mae_values) * 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(EVAL / "model_mae_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 3. Head-to-Head: XGBoost vs Few-Shot

The two best models are remarkably close (MAE 2.11 vs 2.16). For each of the 243 test predictions, which model was closer to the actual points?

In [ ]:
# Head-to-head comparison
xgb_errors = np.abs(predictions["xgboost"]["parsed_prediction"].values - predictions["xgboost"]["actual"].values)
fs_errors = np.abs(predictions["few_shot"]["parsed_prediction"].values - predictions["few_shot"]["actual"].values)

xgb_wins = np.sum(xgb_errors < fs_errors)
fs_wins = np.sum(fs_errors < xgb_errors)
ties = np.sum(xgb_errors == fs_errors)

print(f"XGBoost closer: {xgb_wins} ({xgb_wins/243:.1%})")
print(f"Few-Shot closer: {fs_wins} ({fs_wins/243:.1%})")
print(f"Tied: {ties} ({ties/243:.1%})")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Left: Win comparison bar ---
ax = axes[0]
categories = ["XGBoost\ncloser", "Tied", "Few-Shot\ncloser"]
counts = [xgb_wins, ties, fs_wins]
bar_cols = [COLORS["xgboost"], "#94a3b8", COLORS["few_shot"]]
bars = ax.bar(categories, counts, color=bar_cols, width=0.6, edgecolor="white", linewidth=2)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{count}\n({count/243:.0%})", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_ylabel("Number of Predictions", fontsize=12)
ax.set_title("Per-Prediction Winner", fontsize=14, fontweight="bold")
ax.set_ylim(0, max(counts) * 1.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# --- Right: Scatter of XGBoost pred vs Few-Shot pred ---
ax = axes[1]
xgb_p = predictions["xgboost"]["parsed_prediction"].values
fs_p = predictions["few_shot"]["parsed_prediction"].values
actual = predictions["xgboost"]["actual"].values

scatter = ax.scatter(xgb_p, fs_p, c=actual, cmap="YlOrRd", alpha=0.7, s=30, edgecolors="#666", linewidth=0.3)
cbar = plt.colorbar(scatter, ax=ax, label="Actual Points")

# Diagonal
lims = [min(xgb_p.min(), fs_p.min()) - 0.5, max(xgb_p.max(), fs_p.max()) + 0.5]
ax.plot(lims, lims, "--", color="#666", linewidth=1, alpha=0.6, label="Agreement line")
ax.set_xlabel("XGBoost Prediction", fontsize=12)
ax.set_ylabel("Few-Shot Prediction", fontsize=12)
ax.set_title("XGBoost vs Few-Shot Predictions", fontsize=14, fontweight="bold")
ax.legend(loc="upper left", fontsize=10)
ax.set_xlim(lims)
ax.set_ylim(lims)

plt.tight_layout()
plt.savefig(EVAL / "head_to_head_xgb_fewshot.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 4. Error Distribution Overlay

Comparing the shape of errors for the top three models. XGBoost has the tightest distribution; the fine-tuned model has longer tails due to its discrete prediction values.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))

for model_key, label, color in [
    ("xgboost", "XGBoost", COLORS["xgboost"]),
    ("few_shot", "Few-Shot", COLORS["few_shot"]),
    ("fine_tuned", "Fine-Tuned v3", COLORS["fine_tuned"]),
]:
    df = predictions[model_key]
    errors = df["parsed_prediction"].values - df["actual"].values
    ax.hist(errors, bins=np.arange(-15.5, 16.5, 1), alpha=0.45, color=color, label=f"{label} (MAE {np.mean(np.abs(errors)):.2f})",
            edgecolor=color, linewidth=0.8)

ax.axvline(0, color="#333", linestyle="-", linewidth=1, alpha=0.5)
ax.set_xlabel("Prediction Error (predicted - actual)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Error Distribution: Top 3 Models", fontsize=16, fontweight="bold", pad=15)
ax.legend(fontsize=11, loc="upper right", framealpha=0.9)
ax.set_xlim(-16, 16)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(EVAL / "error_distribution_overlay.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 5. Position-by-Position Breakdown

MAE broken down by player position for the top three models. Forwards are hardest to predict across all approaches -- their points are more volatile (goals, assists, bonus).

In [ ]:
# Compute per-position MAE for all models using test_set position mapping
positions = ["GK", "DEF", "MID", "FWD"]
pos_mae = {}

for model_key in ["xgboost", "few_shot", "fine_tuned"]:
    df = predictions[model_key]
    model_pos_mae = {}
    for pos in positions:
        mask = test_set["position"].values == pos
        if mask.sum() == 0:
            model_pos_mae[pos] = 0
            continue
        errs = np.abs(df["parsed_prediction"].values[mask] - df["actual"].values[mask])
        model_pos_mae[pos] = np.mean(errs)
    pos_mae[model_key] = model_pos_mae

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(positions))
width = 0.25

for i, (model_key, label, color) in enumerate([
    ("xgboost", "XGBoost", COLORS["xgboost"]),
    ("few_shot", "Few-Shot", COLORS["few_shot"]),
    ("fine_tuned", "Fine-Tuned v3", COLORS["fine_tuned"]),
]):
    vals = [pos_mae[model_key][p] for p in positions]
    bars = ax.bar(x + i * width, vals, width, label=label, color=color, edgecolor="white", linewidth=1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f"{val:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xticks(x + width)
ax.set_xticklabels(positions, fontsize=12, fontweight="bold")
ax.set_ylabel("Mean Absolute Error", fontsize=12)
ax.set_title("MAE by Position", fontsize=16, fontweight="bold", pad=15)
ax.legend(fontsize=11, framealpha=0.9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylim(0, max(max(pos_mae[k].values()) for k in pos_mae) * 1.2)

plt.tight_layout()
plt.savefig(EVAL / "position_mae_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 6. The Accuracy vs Effort Trade-off

Development effort (qualitative) plotted against accuracy. The key insight: **few-shot prompting gets most of the way to XGBoost accuracy with near-zero development effort.** Fine-tuning required the most effort (3 iterations to address mode collapse) for worse accuracy than a simple prompt.

In [ ]:
# Effort vs Accuracy scatter
# Effort scale (qualitative): 1=trivial, 2=easy, 3=moderate, 4=significant, 5=extensive
effort_data = {
    "Zero-Shot":        {"effort": 1.0, "mae": 11.54, "color": COLORS["zero_shot"],       "desc": "One prompt, no examples"},
    "Few-Shot":         {"effort": 1.5, "mae": 2.16,  "color": COLORS["few_shot"],        "desc": "3 examples in prompt"},
    "Chain-of-Thought": {"effort": 2.0, "mae": 2.20,  "color": COLORS["chain_of_thought"], "desc": "Structured reasoning prompt"},
    "XGBoost":          {"effort": 3.5, "mae": 2.11,  "color": COLORS["xgboost"],         "desc": "Feature eng + training pipeline"},
    "Fine-Tuned v3":    {"effort": 5.0, "mae": 3.35,  "color": COLORS["fine_tuned"],      "desc": "3 LoRA iterations, mode collapse"},
}

fig, ax = plt.subplots(figsize=(10, 6))

for name, d in effort_data.items():
    ax.scatter(d["effort"], d["mae"], s=200, color=d["color"], edgecolors="white", linewidth=2, zorder=5)
    # Position labels to avoid overlap
    offset_x, offset_y = 0.15, 0.0
    if name == "Zero-Shot":
        offset_y = -0.6
    elif name == "Few-Shot":
        offset_y = 0.25
    elif name == "Chain-of-Thought":
        offset_y = -0.35
    elif name == "XGBoost":
        offset_y = 0.25
    elif name == "Fine-Tuned v3":
        offset_y = 0.25
    ax.annotate(f"{name}\n(MAE {d['mae']:.2f})",
                xy=(d["effort"], d["mae"]),
                xytext=(d["effort"] + offset_x, d["mae"] + offset_y),
                fontsize=10, fontweight="bold", color=d["color"],
                arrowprops=dict(arrowstyle="-", color="#ccc", lw=0.8) if abs(offset_y) > 0.3 else None)

# Shaded "sweet spot" region
from matplotlib.patches import FancyBboxPatch
sweet_spot = FancyBboxPatch((1.2, 1.8), 1.0, 0.7, boxstyle="round,pad=0.1",
                             facecolor=COLORS["few_shot"], alpha=0.08, edgecolor=COLORS["few_shot"],
                             linewidth=1.5, linestyle="--")
ax.add_patch(sweet_spot)
ax.text(1.7, 1.85, "sweet spot", ha="center", fontsize=9, color=COLORS["few_shot"], fontstyle="italic", alpha=0.7)

ax.set_xlabel("Development Effort (qualitative scale)", fontsize=12)
ax.set_ylabel("MAE (lower is better)", fontsize=12)
ax.set_title("Accuracy vs Development Effort", fontsize=16, fontweight="bold", pad=15)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(["Trivial", "Easy", "Moderate", "Significant", "Extensive"], fontsize=10)
ax.set_ylim(0, 13)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(EVAL / "accuracy_vs_effort.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 7. The Ranking Argument

Accuracy is not the only metric that matters for FPL. The real decision task is **ranking**: "Should I start Player A or Player B?" A model that predicts 2.5 for everyone has decent MAE but is useless for decisions.

The fine-tuned model produces 6 unique values {0, 1, 2, 4, 9, 10}, giving a wider spread that differentiates players more clearly. XGBoost predictions cluster in a narrow continuous range. The smoothed ensemble (40% LLM + 30% form + 30% features) produces 66 unique decimal values -- the best of both worlds.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# XGBoost prediction distribution
ax = axes[0]
xgb_p = predictions["xgboost"]["parsed_prediction"].values
ax.hist(xgb_p, bins=30, color=COLORS["xgboost"], alpha=0.8, edgecolor="white", linewidth=0.8)
ax.set_title(f"XGBoost\n{len(np.unique(np.round(xgb_p, 1)))} unique values (1dp)", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Points")
ax.set_ylabel("Count")
pred_range = xgb_p.max() - xgb_p.min()
ax.text(0.95, 0.95, f"Range: {xgb_p.min():.1f} - {xgb_p.max():.1f}\nStd: {xgb_p.std():.2f}",
        transform=ax.transAxes, ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Fine-tuned prediction distribution
ax = axes[1]
ft_p = predictions["fine_tuned"]["parsed_prediction"].values
unique_vals = sorted(np.unique(ft_p))
ax.hist(ft_p, bins=np.arange(-0.5, max(unique_vals) + 1.5, 1), color=COLORS["fine_tuned"], alpha=0.8,
        edgecolor="white", linewidth=0.8)
ax.set_title(f"Fine-Tuned v3\n{len(unique_vals)} unique values {set(int(v) for v in unique_vals)}", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Points")
ax.set_ylabel("Count")
ax.text(0.95, 0.95, f"Range: {ft_p.min():.0f} - {ft_p.max():.0f}\nStd: {ft_p.std():.2f}",
        transform=ax.transAxes, ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Overlay comparison: prediction spread
ax = axes[2]
bp_data = [
    predictions["xgboost"]["parsed_prediction"].values,
    predictions["few_shot"]["parsed_prediction"].values,
    predictions["fine_tuned"]["parsed_prediction"].values,
]
bp_labels = ["XGBoost", "Few-Shot", "Fine-Tuned v3"]
bp_colors = [COLORS["xgboost"], COLORS["few_shot"], COLORS["fine_tuned"]]

bp = ax.boxplot(bp_data, labels=bp_labels, patch_artist=True, widths=0.5,
                medianprops=dict(color="#333", linewidth=2))
for patch, color in zip(bp["boxes"], bp_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_title("Prediction Spread", fontsize=13, fontweight="bold")
ax.set_ylabel("Predicted Points")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(EVAL / "ranking_argument.png", dpi=150, bbox_inches="tight")
plt.show()

The fine-tuned model's wider spread is a feature, not a bug. When choosing between a goalkeeper (predicted 1) and a midfielder (predicted 9), the fine-tuned model gives a clear signal. XGBoost might predict 2.8 vs 3.4 for the same pair -- technically more accurate, but the ranking signal is weaker.

For FPL decisions, **separability matters more than precision**. The ideal system combines XGBoost's calibrated estimates with the fine-tuned model's decisive ranking.

---

## 8. Product Recommendation

### The Hybrid Architecture

No single model wins on every dimension. The production system should combine strengths:

| Use Case | Model | Why |
|----------|-------|-----|
| **Point estimation** | XGBoost | Best MAE (2.11), instant inference, 82.3% within +/-3 |
| **Quick prototyping / new features** | Few-Shot | 97% of XGBoost accuracy with a prompt change |
| **Player ranking / transfer decisions** | Fine-Tuned v3 | Wider spread separates players more clearly |
| **User-facing explanations** | Chain-of-Thought | Visible reasoning builds trust and engagement |
| **Ensemble smoothing** | 40/30/30 blend | 66 unique values; best ranking + calibration |

### When NOT to use the LLM

- **High-volume batch predictions**: 370s for 243 players (few-shot) vs <1s for XGBoost
- **Real-time gameweek updates**: Latency matters; XGBoost is 370x faster
- **Without examples**: Zero-shot MAE of 11.54 is catastrophic; the LLM hallucinates FPL point scales

### When the LLM adds clear value

- **Explaining predictions**: "Salah is predicted 6 points because he is home vs a team conceding 2.1 goals/game, and his 3-GW form is 8.3"
- **Handling novel situations**: New signings, injury returns, schedule changes -- the LLM can reason about context that features miss
- **Rapid iteration**: Testing a new hypothesis takes minutes with prompt engineering vs hours of feature engineering

In [ ]:
# Summary radar/table of model strengths
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis("off")

# Strength comparison matrix
dimensions = ["Accuracy", "Speed", "Ranking\nSeparation", "Explainability", "Dev Effort\n(lower=better)"]
scores = {
    "XGBoost":       [5, 5, 2, 1, 3],
    "Few-Shot":      [4.8, 1, 3, 3, 5],
    "Fine-Tuned v3": [3, 2, 4, 2, 1],
}

cell_text = []
cell_colors = []
for model_name, model_scores in scores.items():
    row = []
    row_colors = []
    for s in model_scores:
        stars = int(round(s))
        row.append("+" * stars + "-" * (5 - stars))
        # Color by score
        intensity = s / 5.0
        g = int(100 + 155 * intensity)
        row_colors.append(f"#{100:02x}{g:02x}{100:02x}20")
    cell_text.append(row)
    cell_colors.append(row_colors)

table = ax.table(
    cellText=cell_text,
    rowLabels=list(scores.keys()),
    colLabels=dimensions,
    cellLoc="center",
    loc="center",
    cellColours=cell_colors,
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

# Style header
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(fontweight="bold")
        cell.set_facecolor("#f1f5f9")
    if j == -1:
        cell.set_text_props(fontweight="bold")
        cell.set_facecolor("#f1f5f9")

ax.set_title("Model Strength Comparison (+ = strong, - = weak)", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

---

## 9. What We Learned

### Five Key Lessons

**1. Data quality matters more than model capacity.**
A 500-tree XGBoost on 17 hand-crafted features beats a fine-tuned LLM with billions of parameters. The features -- form, fixture difficulty, minutes, ICT index -- encode domain knowledge that the LLM must learn from scratch.

**2. Few-shot prompting is shockingly powerful.**
Three examples in a prompt (MAE 2.16) nearly match a trained XGBoost (MAE 2.11). This is the highest ROI approach: minutes of prompt writing vs hours of feature engineering. For any new prediction domain, start here.

**3. Aggregate metrics can be misleading.**
The Chain-of-Thought model has the highest within-1 accuracy (60.1%) but only third-best MAE (2.20). It predicts 0 or 1 for most players -- correct for the many low-scoring players, but useless for identifying high performers. Always examine the full error distribution.

**4. Accuracy is not the same as utility.**
The fine-tuned model has the worst MAE of the serious contenders (3.35), but its 6 unique prediction values spread from 0 to 10 -- a wider range than XGBoost's typical 1-6 output band. For the actual FPL task (ranking players), wider separation can be more useful than tighter calibration.

**5. Build the evaluation framework first.**
Having a structured eval suite (per-position, per-scenario, edge cases) caught problems that aggregate MAE would have hidden -- like the fine-tuned model's mode collapse on v1 and v2, or the Chain-of-Thought model's tendency to predict 0 or 1. The eval infrastructure was the single most valuable engineering investment.

In [ ]:
# Final summary visualization: key numbers
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis("off")

summary_data = [
    ("243", "Test Players", "GW31+"),
    ("2.11", "Best MAE", "XGBoost"),
    ("2.16", "Few-Shot MAE", "Zero training"),
    ("82.3%", "Within +/-3", "XGBoost"),
    ("<1s", "Inference", "XGBoost"),
    ("370s", "Inference", "Few-Shot"),
]

for i, (number, label, sublabel) in enumerate(summary_data):
    x = (i + 0.5) / len(summary_data)
    ax.text(x, 0.7, number, transform=ax.transAxes, ha="center", va="center",
            fontsize=24, fontweight="bold", color=COLORS["xgboost"])
    ax.text(x, 0.35, label, transform=ax.transAxes, ha="center", va="center",
            fontsize=11, color="#333")
    ax.text(x, 0.15, sublabel, transform=ax.transAxes, ha="center", va="center",
            fontsize=9, color="#888", fontstyle="italic")

ax.set_title("Key Numbers at a Glance", fontsize=16, fontweight="bold", pad=15, y=1.0)

# Divider lines
for i in range(1, len(summary_data)):
    x = i / len(summary_data)
    ax.axvline(x, ymin=0.1, ymax=0.9, color="#ddd", linewidth=1)

plt.tight_layout()
plt.savefig(EVAL / "key_numbers.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nNotebook complete. All charts saved to eval/.")